In [2]:
import copy
from heapq import heappush, heappop

# Ma trận kề biểu diễn chi phí di chuyển giữa các thành phố
# (Ví dụ: 4 thành phố đánh số 0, 1, 2, 3)
graph = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]
n = len(graph) # Số lượng thành phố

# ==========================================
# CẤU TRÚC DỮ LIỆU (Tương tự code mẫu trên lớp)
# ==========================================

class priorityQueue:
    def __init__(self):
        self.heap = []

    def push(self, key):
        heappush(self.heap, key)

    def pop(self):
        return heappop(self.heap)

    def is_empty(self):
        return len(self.heap) == 0

class nodes:
    def __init__(self, parent, current_city, visited_cities, g_cost, f_cost):
        self.parent = parent
        self.current_city = current_city
        self.visited_cities = visited_cities

        self.g_cost = g_cost  # g(n): Chi phí thực tế đi từ Start đến hiện tại
        self.f_cost = f_cost  # f(n) = g(n) + h(n)

    def __lt__(self, nxt):
        return self.f_cost < nxt.f_cost

# ==========================================
# CÁC HÀM XỬ LÝ
# ==========================================

# Hàm tính heuristic h(n)
def calculateCosts(current_city, visited_cities, start_city) -> int:
    # Nếu đã thăm tất cả các thành phố, chi phí h(n) là quãng đường quay về điểm xuất phát
    if len(visited_cities) == n:
        return graph[current_city][start_city]

    # Nếu chưa, ước lượng h(n) bằng cạnh ngắn nhất từ thành phố hiện tại tới 1 thành phố chưa thăm
    min_cost = float('inf')
    for i in range(n):
        if i not in visited_cities and graph[current_city][i] > 0:
            if graph[current_city][i] < min_cost:
                min_cost = graph[current_city][i]

    if min_cost == float('inf'):
        return 0
    return min_cost

def newNodes(parent, next_city, current_visited, current_g_cost, start_city) -> nodes:
    # Cập nhật danh sách các thành phố đã đi qua
    new_visited = copy.deepcopy(current_visited)
    new_visited.append(next_city)

    # Cập nhật chi phí thực tế g(n)
    new_g_cost = current_g_cost + graph[parent.current_city][next_city]

    # Tính f(n) = g(n) + h(n)
    h_cost = calculateCosts(next_city, new_visited, start_city)
    new_f_cost = new_g_cost + h_cost

    return nodes(parent, next_city, new_visited, new_g_cost, new_f_cost)

# ==========================================
# VÒNG LẶP CHÍNH CỦA A* (AKT)
# ==========================================
def solve(start_city):
    pq = priorityQueue()

    # Khởi tạo node gốc
    visited = [start_city]
    g_cost = 0
    h_cost = calculateCosts(start_city, visited, start_city)
    f_cost = g_cost + h_cost

    root = nodes(None, start_city, visited, g_cost, f_cost)
    pq.push(root)

    while not pq.is_empty():
        minimum = pq.pop()

        # ĐIỀU KIỆN DỪNG: Đã thăm đủ số thành phố VÀ đã quay lại thành phố xuất phát
        if len(minimum.visited_cities) == n + 1 and minimum.current_city == start_city:
            # Truy vết đường đi
            path = []
            curr = minimum
            while curr:
                path.append(curr.current_city)
                curr = curr.parent
            path.reverse() # Đảo ngược để in từ Start -> End

            print("--- TÌM THẤY CHU TRÌNH GIAO HÀNG TỐI ƯU ---")
            print(f"Hành trình: {' -> '.join(map(str, path))}")
            print(f"Tổng chi phí: {minimum.g_cost}")
            return

        # SINH CÁC TRẠNG THÁI KỀ
        if len(minimum.visited_cities) == n:
            # Nếu đã thăm đủ n thành phố, bắt buộc phải sinh trạng thái quay về điểm xuất phát
            child = newNodes(minimum, start_city, minimum.visited_cities, minimum.g_cost, start_city)
            pq.push(child)
        else:
            # Nếu chưa, duyệt qua các thành phố chưa được thăm
            for next_city in range(n):
                if next_city not in minimum.visited_cities:
                    child = newNodes(minimum, next_city, minimum.visited_cities, minimum.g_cost, start_city)
                    pq.push(child)

# ==========================================
# CHẠY THỬ
# ==========================================
# Bắt đầu xuất phát từ thành phố số 0
solve(0)

--- TÌM THẤY CHU TRÌNH GIAO HÀNG TỐI ƯU ---
Hành trình: 0 -> 1 -> 3 -> 2 -> 0
Tổng chi phí: 80
